In [26]:
import os
import boto3
from botocore import UNSIGNED
from botocore.config import Config

bucket = "stocktwits-nyu"
prefix = "dataset/v1/data/csv/feature_wo_messages/"

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != prefix:
            print(os.path.basename(key))

feature_wo_messages_000.csv
feature_wo_messages_001.csv
feature_wo_messages_002.csv
feature_wo_messages_003.csv
feature_wo_messages_004.csv
feature_wo_messages_005.csv
feature_wo_messages_006.csv
feature_wo_messages_007.csv
feature_wo_messages_008.csv
feature_wo_messages_009.csv
feature_wo_messages_010.csv
feature_wo_messages_011.csv
feature_wo_messages_012.csv
feature_wo_messages_013.csv
feature_wo_messages_014.csv
feature_wo_messages_015.csv
feature_wo_messages_016.csv
feature_wo_messages_017.csv
feature_wo_messages_018.csv
feature_wo_messages_019.csv
feature_wo_messages_020.csv
feature_wo_messages_021.csv
feature_wo_messages_022.csv
feature_wo_messages_023.csv
feature_wo_messages_024.csv
feature_wo_messages_025.csv
feature_wo_messages_026.csv
feature_wo_messages_027.csv
feature_wo_messages_028.csv
feature_wo_messages_029.csv
feature_wo_messages_030.csv
feature_wo_messages_031.csv
feature_wo_messages_032.csv
feature_wo_messages_033.csv
feature_wo_messages_034.csv
feature_wo_messages_

In [27]:
import pandas as pd

In [28]:
BASE_URL = "s3://stocktwits-nyu" # or local path BASE_URL="local_path"
CSV_URL = f"{BASE_URL}/dataset/v1/data/csv"

In [32]:
dtype_map = {"sentiment": "object", "message_id": "object"}
base_url = f"{CSV_URL}/feature_wo_messages"

dfs = {}

for i in range(237, 248):
    url = f"{base_url}/feature_wo_messages_{i:03d}.csv"
    print("attempting to load", url)
    try:
        df = pd.read_csv(url, dtype=dtype_map, usecols=["user_id", "created_at", "sentiment", "symbol_list"])
        name = f"feature_wo_messages_{i:03d}.csv"
        dfs[name] = df
        print("loaded", name)
    except Exception as e:
        print("missing or unreadable:", url, "|", type(e).__name__)
        continue

attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_237.csv
loaded feature_wo_messages_237.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_238.csv
loaded feature_wo_messages_238.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_239.csv
loaded feature_wo_messages_239.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_240.csv
loaded feature_wo_messages_240.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_241.csv
loaded feature_wo_messages_241.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_242.csv
loaded feature_wo_messages_242.csv
attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_243.csv
loaded feature_wo_messages_243.csv
attemp

In [33]:
print("loaded files:", len(dfs))

combined_df = pd.concat(
    [df.assign(source_file=name) for name, df in dfs.items()],
    ignore_index=True
)

loaded files: 11


In [31]:
combined_df

,user_id,created_at,sentiment,symbol_list,source_file
0,727510,2016-07-06T03:05:24Z,NaN,['HES'],feature_wo_messages_227.csv
1,727510,2016-07-06T03:05:24Z,NaN,['SFL'],feature_wo_messages_227.csv
2,727510,2016-07-06T03:05:25Z,NaN,['LEN'],feature_wo_messages_227.csv
3,727510,2016-07-06T03:05:25Z,NaN,['NLS'],feature_wo_messages_227.csv
4,727510,2016-07-06T03:05:25Z,NaN,['HAR'],feature_wo_messages_227.csv
...,...,...,...,...,...
15164948,210967,2017-01-24T01:22:06Z,NaN,[],feature_wo_messages_233.csv
15164949,751731,2017-01-24T01:22:10Z,NaN,[],feature_wo_messages_233.csv
15164950,480560,2017-01-24T01:22:10Z,NaN,[],feature_wo_messages_233.csv
15164951,842546,2017-01-24T01:22:09Z,Bullish,['NUGT'],feature_wo_messages_233.csv


In [34]:
combined_df = combined_df.dropna(subset=["sentiment"])
combined_df = combined_df[combined_df["symbol_list"] != "[]"]

In [ ]:
combined_df["created_at"] = pd.to_datetime(combined_df["created_at"])

In [38]:
combined_df = combined_df[combined_df["created_at"] >= "2016-12-31"]

In [39]:
import ast
combined_df = combined_df.drop(columns=["source_file"])
combined_df["symbol_list"] = combined_df["symbol_list"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

combined_df = combined_df.explode("symbol_list", ignore_index=True)

combined_df = combined_df.rename(columns={"symbol_list": "symbol"})

In [40]:
print(combined_df.head())

   user_id                created_at sentiment symbol
0   960715 2017-03-29 02:18:55+00:00   Bearish   TSLA
1   980760 2017-03-29 02:19:00+00:00   Bullish    BUD
2   170316 2017-03-29 02:19:06+00:00   Bullish   JDST
3   747002 2017-03-29 02:19:09+00:00   Bullish    DIS
4   748375 2017-03-29 02:19:29+00:00   Bullish   SONC


In [41]:
spx_tickers = [
    "A", "AAL", "AAP", "AAPL", "ABBV", "ABC", "ABT", "ACN", "ADBE", "ADI", "ADM", "ADP",
    "ADS", "ADSK", "AEE", "AEP", "AES", "AET", "AFL", "AIG", "AIV", "AIZ", "AJG", "AKAM",
    "ALB", "ALK", "ALL", "ALLE", "ALXN", "AMAT", "AME", "AMG", "AMGN", "AMP", "AMT", "AMZN",
    "AN", "ANDV", "ANTM", "AON", "APA", "APC", "APD", "APH", "APTV", "ATVI", "AVB", "AVGO",
    "AVY", "AWK", "AXP", "AYI", "AZO", "BA", "BAC", "BAX", "BBBY", "BBT", "BBY", "BCR",
    "BDX", "BEN", "BF-B", "BIIB", "BK", "BKNG", "BLK", "BLL", "BMY", "BRK-B", "BSX", "BWA",
    "BXP", "C", "CA", "CAG", "CAH", "CAT", "CB", "CBRE", "CBS", "CCI", "CCL", "CELG",
    "CERN", "CF", "CFG", "CHD", "CHRW", "CHTR", "CI", "CINF", "CL", "CLX", "CMA", "CMCSA",
    "CME", "CMG", "CMI", "CMS", "CNC", "CNP", "COF", "COG", "COL", "COO", "COP", "COST",
    "COTY", "CPB", "CPRI", "CRM", "CSCO", "CSRA", "CSX", "CTAS", "CTL", "CTSH", "CTXS", "CVS",
    "CVX", "CXO", "D", "DAL", "DD", "DE", "DFS", "DG", "DGX", "DHI", "DHR", "DIS",
    "DISCA", "DISCK", "DLR", "DLTR", "DOV", "DRI", "DTE", "DUK", "DVA", "DVN", "EA", "EBAY",
    "ECL", "ED", "EFX", "EIX", "EL", "EMN", "EMR", "ENDP", "EOG", "EQIX", "EQR", "EQT",
    "ES", "ESRX", "ESS", "ETFC", "ETN", "ETR", "EVHC", "EW", "EXC", "EXPD", "EXPE", "EXR",
    "F", "FAST", "FB", "FBHS", "FCX", "FDX", "FE", "FFIV", "FIS", "FISV", "FITB", "FL",
    "FLIR", "FLR", "FLS", "FMC", "FRT", "FSLR", "FTI", "FTR", "FTV", "GD", "GE", "GGP",
    "GILD", "GIS", "GLW", "GM", "GOOG", "GOOGL", "GPC", "GPN", "GPS", "GRMN", "GS", "GT",
    "GWW", "HAL", "HAS", "HBAN", "HBI", "HCA", "HD", "HES", "HIG", "HOG", "HOLX", "HON",
    "HP", "HPE", "HPQ", "HRB", "HRL", "HRS", "HSIC", "HST", "HSY", "HUM", "IBM", "ICE",
    "IFF", "ILMN", "INTC", "INTU", "IP", "IPG", "IRM", "ISRG", "ITW", "IVZ", "JBHT", "JCI",
    "JEC", "JEF", "JNJ", "JNPR", "JPM", "JWN", "K", "KDP", "KEY", "KHC", "KIM", "KLAC",
    "KMB", "KMI", "KMX", "KO", "KORS", "KR", "KSS", "KSU", "L", "LB", "LEG", "LEN",
    "LH", "LKQ", "LLTC", "LLY", "LMT", "LNC", "LNT", "LOW", "LRCX", "LUV", "LVLT", "LYB",
    "M", "MA", "MAA", "MAC", "MAR", "MAS", "MAT", "MCD", "MCHP", "MCK", "MCO", "MDLZ",
    "MDT", "MET", "MHK", "MJN", "MKC", "MLM", "MMC", "MMM", "MNST", "MO", "MOS", "MPC",
    "MRK", "MRO", "MS", "MSFT", "MSI", "MTB", "MTD", "MU", "MUR", "MYL", "NAVI", "NBL",
    "NDAQ", "NEE", "NEM", "NFLX", "NFX", "NI", "NKE", "NLSN", "NOC", "NOV", "NRG", "NSC",
    "NTAP", "NTRS", "NUE", "NVDA", "NWL", "NWS", "NWSA", "O", "OKE", "OMC", "ORCL", "ORLY",
    "OXY", "PAYX", "PBCT", "PBI", "PCAR", "PCG", "PDCO", "PEG", "PEP", "PFE", "PFG", "PG",
    "PGR", "PH", "PHM", "PKI", "PLD", "PM", "PNC", "PNR", "PNW", "PPG", "PPL", "PRGO",
    "PRU", "PSA", "PSX", "PVH", "PWR", "PXD", "PYPL", "QCOM", "QRVO", "R", "RAI", "RCL",
    "REGN", "RF", "RHI", "RHT", "RL", "ROK", "ROP", "ROST", "RRC", "RSG", "RTN", "SBUX",
    "SCG", "SCHW", "SEE", "SHW", "SJM", "SLB", "SLG", "SNA", "SNI", "SO", "SPG", "SPGI",
    "SPLS", "SRCL", "SRE", "STI", "STJ", "STT", "STX", "STZ", "SWK", "SWKS", "SWN", "SYF",
    "SYK", "SYMC", "SYY", "T", "TAP", "TDC", "TDG", "TEL", "TGNA", "TGT", "TIF", "TJX",
    "TMK", "TMO", "TPR", "TRIP", "TROW", "TRV", "TSCO", "TSN", "TSS", "TWX", "TXN", "TXT",
    "UA", "UAA", "UAL", "UDR", "UHS", "ULTA", "UNH", "UNM", "UNP", "UPS", "URI", "USB",
    "UTX", "V", "VAR", "VFC", "VIAB", "VLO", "VMC", "VNO", "VRSK", "VRSN", "VRTX", "VTR",
    "VZ", "WAT", "WBA", "WDC", "WEC", "WELL", "WFC", "WFM", "WHR", "WLTW", "WM", "WMB",
    "WMT", "WRK", "WU", "WY", "WYNN", "XEC", "XEL", "XLNX", "XOM", "XRAY", "XRX", "XYL",
    "YUM", "ZBH", "ZION", "ZTS"
]

In [42]:
df_snp = combined_df[combined_df["symbol"].isin(spx_tickers)]

In [ ]:
df_snp.shape

(451297, 4)

In [44]:
df_snp["sentiment"].value_counts()

sentiment
Bullish    332240
Bearish    119057
Name: count, dtype: int64

In [49]:
len(df_snp.symbol.unique())

361

In [50]:
df_snp

,user_id,created_at,sentiment,symbol
4,12134,2011-11-14 19:14:55+00:00,Bearish,VLO
6,32357,2011-11-14 19:37:06+00:00,Bullish,CBOE
12,42105,2011-11-14 21:06:02+00:00,Bullish,TXN
35,16229,2011-11-15 02:43:22+00:00,Bullish,DPZ
38,3202,2011-11-15 03:19:40+00:00,Bearish,LVS
...,...,...,...,...
1130863,276143,2017-11-02 22:07:18+00:00,Bullish,GOOGL
1130866,1242344,2017-11-02 22:07:45+00:00,Bullish,AAPL
1130877,441500,2017-11-02 22:08:37+00:00,Bullish,AAPL
1130879,547349,2017-11-02 22:08:50+00:00,Bullish,AAPL


In [46]:
df_snp = (
    df_snp.sort_values("created_at")
      .drop_duplicates(subset=["user_id", "symbol"], keep="last")
      .reset_index(drop=True)
)

In [48]:
df_snp = pd.read_csv("cleaned_data.csv", index_col=0, parse_dates=["created_at"])